# ER Patient Triage Optimization

| Field | Value |
|---|---|
| **Author** | Zachary Failor |
| **Course** | ADS 5834 — Data Science in Practice |
| **Notebook** | data_understanding |
| **Created Date** | 2026-11-10 |
| **Last Updated Date** | 2026-11-10 |
| **Last Updated By** | Zachary Failor |
| **Technical Owner Email** | zachf@vt.edu |
| **Business Owner Email** | zachf@vt.edu |

## Project Description
Predict patient severity and optimize assignments to beds, physicians, and time slots to improve care effectiveness while respecting hospital resource limits.

## Notebook Description
Explore and prepare the patient and hospital resource data, describe the tables and their relationships, visualize important patterns, and check data quality before preparing the data for modeling. 

## 2.1 Collect Initial Data
The data was taken from workspace.healthcare_triage in Databricks using Spark. Seven tables loaded successfully.

We need patient information and known severity scores to train the model. For scheduling, we need arrival times, beds, physicians, their capacities, care quality scores, and time settings. The tables have this information. 

All seven tables were selected for inspection. Patient characteristics will be used as predictors and severity_score is the target. Patient_id will only identify records. We will use all 4,000 historical records because the historical table has no date column for selecting a shorter time.

Sex and chief complaint contain categories rather than open-ended text and will need encoding before modeling. Arrival times will need to be turned into timestamps. We will check that their category names match before combining tables.

If additional required information is missing, we will ask the company whether it is available.

In [0]:
table_names = [
    "er_patient_history",
    "er_scheduling_input",
    "er_resources_beds",
    "er_resources_physicians",
    "er_resources_care_quality",
    "er_resources_time_config",
    "er_optimization_metrics"
]

tables = {}

for name in table_names:
    df = spark.read.table(f"workspace.healthcare_triage.{name}")
    tables[name] = df
    print(f"{name}: {df.count()} rows, {len(df.columns)} columns")

### Dataset List

| Table | Rows | Columns |
|---|---:|---:|
| er_patient_history | 4,000 | 16 |
| er_scheduling_input | 177 | 16 |
| er_resources_beds | 25 | 3 |
| er_resources_physicians | 6 | 3 |
| er_resources_care_quality | 6 | 3 |
| er_resources_time_config | 4 | 2 |
| er_optimization_metrics | 1 | 9 |

The scheduling table has 177 rows. However, there are 120 patients described in the project instructions. This difference needs further investigation.

## 2.2 Describe Data

### Patient Data
- **er_patient_history:** Contains 4,000 records and 16 columns. Each row represents a historical patient with demographics, chief complaint, vital signs, existing conditions, ambulance arrival, and a known severity score.
- **er_scheduling_input:** Contains 177 records and 16 columns. It includes the same patient features, but replaces severity_score with arrival_time. These patients need severity predictions before scheduling.

The prediction target is the severity_score and it ranges from 1–5. Patient IDs identify records and will not be used as predictors.

### Hospital Data
- **er_resources_beds:** Contains 25 beds, identified by bed_id, with bed type and capacity.
- **er_resources_physicians:** Contains 6 physicians: 2 attendings and 4 residents. Each attending can handle 2 patients per slot, while each resident can handle 1.
- **er_resources_care_quality:** Contains 6 bed-type and physician-type combinations with care quality multipliers ranging from 0.8–1.0.
- **er_resources_time_config:** Defines 24 slots lasting 30 minutes each, covering 12 hours starting at 6 PM.
- **er_optimization_metrics:** Contains one existing summary of scheduling metrics. Its source and calculation methods need to be verified before using it as a comparison baseline.

### Data Types
Patient IDs, sex, and chief complaints are stored as strings. Most numeric fields are integers, while temperature and care quality multipliers are doubles. Arrival time is currently stored as a string.

### Table Relationships
The care quality table connects to beds through bed_type and to physicians through physician_type. This corresponds to the physicians table's type column.

Historical and scheduling patient records serve separate purposes and do not need to be joined together. The scheduling process will combine patient severity predictions with the resource and time configuration tables.

In [0]:
history = tables["er_patient_history"]

history.printSchema()
display(history.limit(10))

In [0]:
for name in table_names:
    if name != "er_patient_history":
        print(f"\n{name}")
        tables[name].printSchema()
        display(tables[name].limit(10))

In [0]:
# Historical patient statistics
display(history.describe())

# Scheduling patient statistics
display(
    tables["er_scheduling_input"]
    .drop("patient_id", "arrival_time")
    .describe()
)

# Number of beds of each type
display(
    tables["er_resources_beds"]
    .groupBy("bed_type")
    .count()
    .orderBy("bed_type")
)

### Summary Statistics Findings

Patients range from 18–95 years old and has an average age of 54.65. Scheduling patients have the same age range and an average age of 55.81.

Patient severity scores range from 1–5, with an average of 3.36. Pain scores range from 0–10 in both datasets. The condition and ambulance arrival indicators range from 0–1.

The scheduling data has a lower average heart rate (94.37 compared with 98.79) and a slightly higher average oxygen saturation (93.92 compared with 93.30). 

All columns included in the summaries have complete counts (they arent null). Scheduling patient IDs and arrival times were not included in these summaries.

The bed inventory is 2 ICU beds, 8 high-acuity beds, and 15 standard beds.

In [0]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(17, 14))
ax.set_xlim(0, 17)
ax.set_ylim(0, 14)
ax.axis("off")

keys = {
    "er_patient_history": ["patient_id"],
    "er_scheduling_input": ["patient_id"],
    "er_resources_beds": ["bed_id"],
    "er_resources_physicians": ["physician_id"],
    "er_resources_care_quality": ["bed_type", "physician_type"],
    "er_resources_time_config": ["parameter"]
}

def draw_table(name, x, top, width=5):
    fields = tables[name].dtypes
    height = 0.6 + len(fields) * 0.27

    ax.add_patch(Rectangle(
        (x, top - height), width, height,
        facecolor="#f1f5f9", edgecolor="#334155"
    ))
    ax.add_patch(Rectangle(
        (x, top - 0.45), width, 0.45,
        facecolor="#24476b", edgecolor="#334155"
    ))

    ax.text(
        x + 0.12, top - 0.23, name,
        color="white", weight="bold", fontsize=10, va="center"
    )

    for i, (field, dtype) in enumerate(fields):
        key_label = " [PK]" if field in keys.get(name, []) else ""
        ax.text(
            x + 0.12, top - 0.65 - i * 0.27,
            f"{field}: {dtype}{key_label}",
            fontsize=9, va="center"
        )

# Patient tables and time configuration
draw_table("er_patient_history", 0.3, 13.2)
draw_table("er_scheduling_input", 6, 13.2)
draw_table("er_resources_time_config", 11.7, 13.2)

# Resource tables
draw_table("er_resources_beds", 0.3, 6.8)
draw_table("er_resources_care_quality", 6, 6.8)
draw_table("er_resources_physicians", 11.7, 6.8)

# Category-based relationships
ax.plot([5.3, 6], [6, 6], color="#334155")
ax.text(
    5.65, 7.05, "bed_type match",
    ha="center", fontsize=9
)

ax.plot([11, 11.7], [6, 6], color="#334155")
ax.text(
    11.35, 7.05, "physician_type = type",
    ha="center", fontsize=9
)

# Standalone summary table
draw_table("er_optimization_metrics", 6, 4.4)

ax.set_title(
    "Healthcare Triage — Entity Relationship Diagram",
    fontsize=16, weight="bold", pad=15
)

fig.text(
    0.5, 0.04,
    "PK = logical key; care quality uses a composite key.\n"
    "Lines show category matches, not declared foreign keys. "
    "Both category relationships are many-to-many.\n"
    "The patient, time configuration, and metrics tables have "
    "no direct join keys to the other source tables.",
    ha="center", fontsize=10
)

plt.tight_layout(rect=[0, 0.1, 1, 1])
plt.show()

### Entity Relationship Diagram
The diagram above shows the seven tables and their columns. The beds and physicians tables connect to the care quality table through their bed and physician types. The historical patients, scheduling patients, time settings, and existing metrics are shown separately because they do not have fields that directly link their records to the other tables.

### Field Meanings

| Field | Meaning |
|---|---|
| patient_id | Identifies a patient record; not used as a predictor |
| age | Patient age in years |
| sex | Recorded sex category |
| chief_complaint | Main reason for the ER visit |
| heart_rate | Heart rate measurement |
| bp_systolic | Systolic blood pressure |
| bp_diastolic | Diastolic blood pressure |
| temperature | Body temperature |
| respiratory_rate | Breathing rate |
| o2_sat | Oxygen saturation |
| pain_score | Recorded pain level from 0–10 |
| diabetes | Diabetes indicator: 1 = yes, 0 = no |
| hypertension | Hypertension indicator: 1 = yes, 0 = no |
| heart_disease | Heart disease indicator: 1 = yes, 0 = no |
| ambulance_arrival | Ambulance arrival indicator: 1 = yes, 0 = no |
| severity_score | Prediction target from 1–5; scores 4–5 represent high severity |
| arrival_time | Patient arrival date and time, used to prevent scheduling before arrival |
| bed_id / physician_id | Identify individual resources |
| bed_type / type | Bed category or physician category |
| capacity | Patient capacity of a bed |
| max_patients_per_slot | Number of patients a physician can handle per time slot |
| care_quality_multiplier | Score used with severity to calculate care effectiveness |
| parameter / value | Scheduling settings and their values |

### Assumptions

- **Patient count:** The scheduling table has 177 patients, the instructions describe 120. For now, we will use all 177 patients.
- **High severity:** We will treat severity scores of 4–5 as high severity, following the project instructions.
- **Capacity:** Physician's can handle 8 patients per slot, or 192 assignments across 24 slots. Scheduling 177 patients would use about 92% of the physician capacity. This assumes each patient uses one slot and all physicians remain available. Arrival times or other constraints could prevent everyone from being scheduled.



## 2.3 Explore Data

This part explores patient severity, the relationship between chief complaints and severity, and patient arrival patterns. These patterns will help with severity prediction and resource scheduling.

In [0]:
import matplotlib.pyplot as plt

severity_counts = (
    history.groupBy("severity_score")
    .count()
    .orderBy("severity_score")
    .toPandas()
)

plt.figure(figsize=(7, 4))
plt.bar(severity_counts["severity_score"], severity_counts["count"])
plt.xlabel("Severity Score")
plt.ylabel("Number of Patients")
plt.title("Historical Patient Severity Distribution")
plt.xticks([1, 2, 3, 4, 5])
plt.tight_layout()
plt.show()

display(severity_counts)

In [0]:
complaint_severity = (
    history.groupBy("chief_complaint", "severity_score")
    .count()
    .toPandas()
)

complaint_percentages = (
    complaint_severity
    .pivot(index="chief_complaint", columns="severity_score", values="count")
    .fillna(0)
    .reindex(columns=[1, 2, 3, 4, 5], fill_value=0)
)

complaint_percentages = (
    complaint_percentages
    .div(complaint_percentages.sum(axis=1), axis=0) * 100
)

complaint_percentages.plot(
    kind="bar", stacked=True, figsize=(9, 5), colormap="viridis"
)
plt.xlabel("Chief Complaint")
plt.ylabel("Percentage of Patients")
plt.title("Severity Distribution by Chief Complaint")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Severity", bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

display(complaint_percentages.round(1))

In [0]:
import pandas as pd

arrivals = (
    tables["er_scheduling_input"]
    .select("arrival_time")
    .toPandas()
)

arrivals["arrival_timestamp"] = pd.to_datetime(
    arrivals["arrival_time"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

print(
    "Missing or unreadable arrival times:",
    arrivals["arrival_timestamp"].isna().sum()
)

hourly_arrivals = (
    arrivals.dropna(subset=["arrival_timestamp"])
    .set_index("arrival_timestamp")
    .resample("h")
    .size()
)

ax = hourly_arrivals.plot(kind="bar", figsize=(10, 4))
ax.set_xticklabels(
    hourly_arrivals.index.strftime("%m/%d %I %p"),
    rotation=45,
    ha="right"
)
plt.xlabel("Arrival Hour")
plt.ylabel("Number of Patients")
plt.title("Scheduling Patient Arrivals by Hour")
plt.tight_layout()
plt.show()

display(hourly_arrivals.rename("patient_count").reset_index())

In [0]:
# Explore numeric relationships
columns = [
    "age",
    "heart_rate",
    "o2_sat",
    "pain_score",
    "ambulance_arrival",
    "severity_score"
]

numeric_data = history.select(columns).toPandas()
correlations = numeric_data.corr()

fig, ax = plt.subplots(figsize=(8, 6))

heatmap = ax.imshow(
    correlations, cmap="coolwarm", vmin=-1, vmax=1
)

ax.set_xticks(range(len(columns)))
ax.set_yticks(range(len(columns)))
ax.set_xticklabels(columns, rotation=45, ha="right")
ax.set_yticklabels(columns)

for row in range(len(columns)):
    for column in range(len(columns)):
        ax.text(
            column, row,
            f"{correlations.iloc[row, column]:.2f}",
            ha="center", va="center", fontsize=9
        )

fig.colorbar(heatmap, ax=ax, label="Pearson correlation")
ax.set_title("Relationships Between Selected Patient Variables")
plt.tight_layout()
plt.show()

### Exploration Findings

Severity 3 is the most common score, with 1,176 patients (29.4%). Severity 1 is the least common, with 240 patients (6%). The model evaluation should check performance for each class, and not just accuracy alone because the severity groups are uneven.

Severity 4–5 patients account for 1,837 of the 4,000 historical records (45.9%). They make up about 64% of neurological complaints and 59% of chest-pain complaints. On the other hand, about 22% of patients in the “other” complaint group have scores of 4 or 5. This suggests that chief complaint could help predict severity with other patient information.

The busiest arrival hour is 9–10 PM, with 28 records. The quietest is 11 PM–midnight, with 7. These uneven arrivals may create periods when demand exceeds available resources.

Patients with higher heart rates and lower oxygen levels tended to have higher severity scores with correlations of 0.76 and −0.76. Also, higher pain scores were connected to higher severity (0.50). These variables may be useful for predicting severity.

These results show that chief complaint may be able to help predict severity when combined with vital signs or other patient information. We will check performance for each severity level, especially scores 4–5 because severity groups are uneven. Also, the busiest arrival periods could lead to longer waits, so we will examine whether the scheduling model can handle that situation.

## 2.4 Verify Data Quality

This part checks for missing values, duplicate records, invalid patient values, and arrival-time problems. Any issues found will guide data preparation.

In [0]:
import pandas as pd
from pyspark.sql import functions as F

missing_results = []
duplicate_results = []

keys = {
    "er_patient_history": ["patient_id"],
    "er_scheduling_input": ["patient_id"],
    "er_resources_beds": ["bed_id"],
    "er_resources_physicians": ["physician_id"],
    "er_resources_care_quality": ["bed_type", "physician_type"],
    "er_resources_time_config": ["parameter"]
}

for name, df in tables.items():
    # Count nulls, blank strings, and numeric NaN values
    checks = []

    for column, dtype in df.dtypes:
        missing = F.col(column).isNull()

        if dtype == "string":
            missing = missing | (F.trim(F.col(column)) == "")
        elif dtype in ["float", "double"]:
            missing = missing | F.isnan(F.col(column))

        checks.append(
            F.coalesce(
                F.sum(F.when(missing, 1).otherwise(0)),
                F.lit(0)
            ).alias(column)
        )

    counts = df.agg(*checks).first().asDict()

    for column, count in counts.items():
        missing_results.append((name, column, count))

    # Check repeated rows and repeated keys separately
    total = df.count()
    duplicate_rows = total - df.dropDuplicates().count()

    duplicate_keys = (
        total - df.dropDuplicates(keys[name]).count()
        if name in keys else None
    )

    duplicate_results.append(
        (name, total, duplicate_rows, duplicate_keys)
    )

missing_report = pd.DataFrame(
    missing_results,
    columns=["table", "column", "missing_values"]
)

print("Missing values by table:")
display(
    missing_report.groupby("table", as_index=False)["missing_values"].sum()
)

missing_columns = missing_report[missing_report["missing_values"] > 0]

if missing_columns.empty:
    print("No missing values were found in any table.")
else:
    print("Columns with missing values:")
    display(missing_columns)

print("Duplicate rows and keys:")
display(pd.DataFrame(
    duplicate_results,
    columns=["table", "total_rows", "duplicate_rows", "duplicate_keys"]
))

In [0]:
quality_results = []

for name in ["er_patient_history", "er_scheduling_input"]:
    df = tables[name]

    rules = {
        "Age outside 0–120": ~F.col("age").between(0, 120),
        "Pain score outside 0–10": ~F.col("pain_score").between(0, 10),
        "Oxygen saturation outside 0–100": ~F.col("o2_sat").between(0, 100),
        "Nonpositive heart rate": F.col("heart_rate") <= 0,
        "Nonpositive respiratory rate": F.col("respiratory_rate") <= 0,
        "Nonpositive blood pressure": (
            (F.col("bp_systolic") <= 0) |
            (F.col("bp_diastolic") <= 0)
        ),
        "Systolic below diastolic": (
            F.col("bp_systolic") < F.col("bp_diastolic")
        )
    }

    for column in [
        "diabetes", "hypertension", "heart_disease", "ambulance_arrival"
    ]:
        rules[f"{column} outside 0/1"] = ~F.col(column).isin(0, 1)

    if name == "er_patient_history":
        rules["Severity outside 1–5"] = ~F.col("severity_score").between(1, 5)

    counts = df.agg(*[
        F.sum(F.when(condition, 1).otherwise(0)).alias(label)
        for label, condition in rules.items()
    ]).first().asDict()

    for label, count in counts.items():
        quality_results.append((name, label, count))

display(pd.DataFrame(
    quality_results,
    columns=["table", "check", "flagged_rows"]
))

# Check arrivals against the supplied Friday-night scenario
arrival_check = tables["er_scheduling_input"].toPandas()

arrival_check["parsed_arrival"] = pd.to_datetime(
    arrival_check["arrival_time"],
    format="%m/%d/%Y %H:%M",
    errors="coerce"
)

start = pd.Timestamp("2024-03-15 18:00")
end = pd.Timestamp("2024-03-16 06:00")

outside_window = (
    (arrival_check["parsed_arrival"] < start) |
    (arrival_check["parsed_arrival"] >= end)
)

display(pd.DataFrame({
    "check": [
        "Missing or unreadable arrival times",
        "Arrivals outside 6 PM–6 AM",
        "Earliest arrival",
        "Latest arrival"
    ],
    "result": [
        str(arrival_check["parsed_arrival"].isna().sum()),
        str(outside_window.sum()),
        str(arrival_check["parsed_arrival"].min()),
        str(arrival_check["parsed_arrival"].max())
    ]
}))

In [0]:
# Check category matches between resources and care quality
beds = tables["er_resources_beds"].select("bed_type").distinct()
quality_beds = tables["er_resources_care_quality"].select("bed_type").distinct()

physicians = (
    tables["er_resources_physicians"]
    .select(F.col("type").alias("physician_type"))
    .distinct()
)
quality_physicians = (
    tables["er_resources_care_quality"]
    .select("physician_type")
    .distinct()
)

checks = [
    ("Bed types missing from care quality", beds.subtract(quality_beds).count()),
    ("Care quality bed types absent from beds", quality_beds.subtract(beds).count()),
    ("Physician types missing from care quality",
     physicians.subtract(quality_physicians).count()),
    ("Care quality physician types absent from physicians",
     quality_physicians.subtract(physicians).count())
]

display(spark.createDataFrame(checks, ["check", "unmatched_categories"]))

### Data Quality Findings

No nulls, blank strings, or missing numeric values were found in the seven tables. No duplicate rows were found, and all checked table keys were unique.

The scheduling table has 177 unique patient IDs. This means the difference from the 120 patients described in the instructions is not caused by duplicate IDs. This needs clarification to ensure accuracy.

One historical patient record has systolic blood pressure below diastolic blood pressure. This record could be misleading during training. It needs to be looked into before deciding how to handle it during data preparation. We should verify the values with the data provider. All other implemented patient-value checks returned zero flagged rows.

All scheduling arrival times were readable and within the intended overnight window. The earliest arrival was March 15, 2024 at 6:00 PM, and the latest was March 16, 2024 at 5:56 AM.

The bed and physician types match those in the care quality table, with no missing or unmatched categories found.

These checks found no need for missing-value fixes or removal of duplicates, but they do not guarantee that every value is correct. It is only based on patients aged 18–95 and should not support predictions for children.